In [2]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.utils import to_categorical

In [5]:
# 문자 기반 Seq2Seq

chars = '0123456789+ '
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

num_chars = len(chars)
print(num_chars)
print(char_to_idx)
print(idx_to_char)

12
{'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '+': 10, ' ': 11}
{0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: '+', 11: ' '}


In [9]:
# 학습 데이터 만들기
def generate_data(n_samples=1000, max_digits=3):
  questions = []
  answers = []

  for _ in range(n_samples):
    a = np.random.randint(0, 10**max_digits)
    b = np.random.randint(0, 10**max_digits)

    q = f"{a}+{b}"
    a_str = str(a + b)

    questions.append(q)
    answers.append(a_str)

  return questions, answers

questions, answers = generate_data()

print(f"질문 : {questions}")
print(f"답변 : {answers}")

질문 : ['567+705', '361+651', '950+894', '155+125', '290+100', '407+150', '877+488', '323+250', '305+544', '848+29', '665+81', '30+284', '941+920', '125+261', '887+209', '928+683', '704+67', '442+111', '611+39', '629+98', '116+707', '651+86', '264+97', '491+102', '646+257', '614+359', '720+324', '242+28', '767+239', '367+305', '711+458', '586+923', '758+686', '892+658', '331+192', '578+372', '960+181', '798+854', '378+902', '998+298', '83+889', '100+540', '87+921', '420+348', '687+560', '333+530', '202+373', '199+28', '776+199', '413+146', '478+605', '268+412', '532+976', '859+901', '309+407', '447+85', '96+924', '810+316', '750+318', '992+515', '69+941', '851+281', '583+840', '996+603', '704+818', '967+813', '37+533', '487+360', '584+714', '845+400', '753+301', '59+58', '358+69', '150+503', '777+350', '809+526', '49+385', '21+354', '183+851', '798+337', '677+974', '690+399', '526+203', '960+25', '558+163', '798+79', '944+112', '428+722', '863+786', '234+687', '229+435', '816+154', '70+3

In [25]:
# 문자 -> OneHot 시퀸스 변환
# seq2seq 입력은 항상 3차원 : (samples, time_steps, vocab_size)

max_q_len = max(len(q) for q in questions)
max_a_len = max(len(a) for a in answers)

def encode_text(texts, max_len):
  data = np.zeros((len(texts), max_len, num_chars))
  for i, text in enumerate(texts):
    for t, char in enumerate(text):
      data[i, t, char_to_idx[char]] = 1
  return data

encoder_input = encode_text(questions, max_q_len)
decoder_target = encode_text(answers, max_a_len)

print(f"encoder_input : {encoder_input.shape}")
print(f"decoder_target : {decoder_target.shape}")

encoder_input : (1000, 7, 12)
decoder_target : (1000, 4, 12)


In [26]:
# Decoder 입력 만들기 (Teacher Forcing)
# Decoder는 정답을 한 칸 오른쪽으로 밀어서 입력을 사용

decoder_input = np.zeros_like(decoder_target)
decoder_input[:, 1:, :] = decoder_target[:, :-1, :]

print(f"decoder_input : {decoder_input[0]}")

decoder_input : [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]]


In [29]:
# Encoder 모델 구성
# 입력 시퀸스를 하나의 context vector(State)로 요약

latent_dim = 128

encoder_inputs = Input(shape=(None, num_chars))
encoder_lstm = LSTM(latent_dim, return_state=True)

_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]


# Decoder 모델 구성
# Encoder state를 초기 상태로 전달
# return_sequence=True 필수
decoder_inputs = Input(shape=(None, num_chars))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

decoder_dense = Dense(num_chars, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Seq2Seq 학습 모델 생성
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, None, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, None, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ [(None, 128),     │     72,192 │ input_layer_10[0… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ [(None, None,     │     72,192 │ input_layer_11[0… │
│                     │ 128), (None,      │            │ lstm_10[0][1],    │
│                     │ 128), (None,      │            │ lstm_10[0][2]     │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, None, 12)  │      1,548 │ lstm_11[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 145,932 (570.05 KB)

 Trainable params: 145,932 (570.05 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
# 모델 학습
model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=32,
    epochs=20,
    validation_split=0.2
)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 37ms/step - accuracy: 0.1875 - loss: 2.1342 - val_accuracy: 0.2375 - val_loss: 1.9128
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2812 - loss: 1.9106 - val_accuracy: 0.2512 - val_loss: 1.8705
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2692 - loss: 1.8797 - val_accuracy: 0.3113 - val_loss: 1.8583
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2652 - loss: 1.8677 - val_accuracy: 0.3338 - val_loss: 1.8431
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2661 - loss: 1.8522 - val_accuracy: 0.2125 - val_loss: 1.8366
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2374 - loss: 1.8350 - val_accuracy: 0.2150 - val_loss: 1.8291
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2812 - loss: 1.8245 - val_accuracy: 0.2100 - val_loss: 1.8327
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2272 - loss: 1.8245 - val_accuracy: 0.2175 - v

In [34]:
# 추론용 Inference 모델 구성

# Encoder
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

In [32]:
# 문장 생성 함수
def decode_sequence(input_seq):
    states = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1, num_chars))

    decoded = ''

    for _ in range(max_a_len):
        output_tokens, h, c = decoder_model.predict([target_seq] + states)

        sampled_idx = np.argmax(output_tokens[0, -1, :])
        sampled_char = idx_to_char[sampled_idx]

        decoded += sampled_char

        target_seq = np.zeros((1, 1, num_chars))
        target_seq[0, 0, sampled_idx] = 1
        states = [h, c]

    return decoded

In [35]:
# 테스트
test_idx = np.random.randint(len(questions))
input_seq = encoder_input[test_idx:test_idx+1]

print("입력 :", questions[test_idx])
print("출력 :", decode_sequence(input_seq))
print("정답 :", answers[test_idx])

입력 : 708+152
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
출력 : 1100
정답 : 860
